In [ ]:
# ============================================================
# Notebook 3: Template validation and injections
# Block 1
#
# What this block does:
# 1. Imports packages
# 2. Defines constants and plotting settings
# 3. Defines the Kerr remnant model
# 4. Defines the qnm-based Kerr QNM lookup
# 5. Defines GR and modified-gravity waveform builders
# ============================================================

from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import qnm


# Physical constants
G = 6.67430e-11
C = 299_792_458.0
MSUN = 1.988409870698051e30

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


# ------------------------------------------------------------
# Kerr remnant
# ------------------------------------------------------------
@dataclass
class KerrRemnant:
    mass_msun: float
    chi: float

    def __post_init__(self):
        if self.mass_msun <= 0:
            raise ValueError("mass_msun must be positive")
        if not (-1.0 < self.chi < 1.0):
            raise ValueError("chi must satisfy -1 < chi < 1")

    @property
    def mass_si(self) -> float:
        return self.mass_msun * MSUN

    @property
    def time_unit(self) -> float:
        return G * self.mass_si / C**3


# ------------------------------------------------------------
# qnm backend
# ------------------------------------------------------------
def get_kerr_qnm(remnant: KerrRemnant, l: int, m: int, n: int, s: int = -2) -> dict:
    mode_seq = qnm.modes_cache(s=s, l=l, m=m, n=n)
    result = mode_seq(a=remnant.chi)
    omega = result[0]

    omega_r_geom = float(np.real(omega))
    omega_i_geom = float(-np.imag(omega))  # qnm returns omega_R - i omega_I

    M_sec = remnant.time_unit
    omega_r_si = omega_r_geom / M_sec
    omega_i_si = omega_i_geom / M_sec

    return {
        "l": l,
        "m": m,
        "n": n,
        "omega_complex_geom": omega,
        "omega_r_geom": omega_r_geom,
        "omega_i_geom": omega_i_geom,
        "omega_r_si": omega_r_si,
        "omega_i_si": omega_i_si,
        "f_hz": omega_r_si / (2.0 * np.pi),
        "tau_s": 1.0 / omega_i_si,
    }


# ------------------------------------------------------------
# Waveform data structures
# ------------------------------------------------------------
@dataclass
class RingdownMode:
    l: int
    m: int
    n: int
    amplitude: float
    phase: float


@dataclass
class ModeDeviation:
    d_omega_frac: float = 0.0
    d_tau_frac: float = 0.0
    d_amp_frac: float = 0.0
    d_phase: float = 0.0


@dataclass
class RingdownSignal:
    t: np.ndarray
    h_complex: np.ndarray

    @property
    def h_plus(self) -> np.ndarray:
        return np.real(self.h_complex)

    @property
    def h_cross(self) -> np.ndarray:
        return -np.imag(self.h_complex)


# ------------------------------------------------------------
# Core low-level waveform builder
# ------------------------------------------------------------
def component_strain_from_freqs(
    amplitude: float,
    phase: float,
    omega_r_si: float,
    omega_i_si: float,
    t: np.ndarray,
    t0: float = 0.0,
) -> np.ndarray:
    dt = t - t0
    mask = dt >= 0.0

    h = np.zeros_like(t, dtype=np.complex128)
    h[mask] = (
        amplitude
        * np.exp(-omega_i_si * dt[mask])
        * np.exp(-1j * (omega_r_si * dt[mask] + phase))
    )
    return h


def get_mode_deviation(
    deviations: dict[tuple[int, int, int], ModeDeviation],
    l: int,
    m: int,
    n: int,
) -> ModeDeviation:
    return deviations.get((l, m, n), ModeDeviation())


def deform_qnm_physical_parameters(
    omega_r_si: float,
    omega_i_si: float,
    deviation: ModeDeviation,
) -> tuple[float, float]:
    if 1.0 + deviation.d_tau_frac <= 0.0:
        raise ValueError("d_tau_frac must satisfy 1 + d_tau_frac > 0")

    omega_r_new = omega_r_si * (1.0 + deviation.d_omega_frac)
    omega_i_new = omega_i_si / (1.0 + deviation.d_tau_frac)

    return omega_r_new, omega_i_new


# ------------------------------------------------------------
# GR and MG waveform generators
# ------------------------------------------------------------
def generate_gr_ringdown(
    remnant: KerrRemnant,
    modes: list[RingdownMode],
    duration: float,
    sample_rate: float,
    t0: float = 0.0,
    s: int = -2,
) -> RingdownSignal:
    if duration <= 0:
        raise ValueError("duration must be positive")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive")

    t = np.arange(0.0, duration, 1.0 / sample_rate)
    h = np.zeros_like(t, dtype=np.complex128)

    for mode in modes:
        q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n, s=s)
        h += component_strain_from_freqs(
            amplitude=mode.amplitude,
            phase=mode.phase,
            omega_r_si=q["omega_r_si"],
            omega_i_si=q["omega_i_si"],
            t=t,
            t0=t0,
        )

    return RingdownSignal(t=t, h_complex=h)


def generate_modified_ringdown(
    remnant: KerrRemnant,
    modes: list[RingdownMode],
    deviations: dict[tuple[int, int, int], ModeDeviation],
    duration: float,
    sample_rate: float,
    t0: float = 0.0,
    s: int = -2,
) -> RingdownSignal:
    if duration <= 0:
        raise ValueError("duration must be positive")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive")

    t = np.arange(0.0, duration, 1.0 / sample_rate)
    h = np.zeros_like(t, dtype=np.complex128)

    for mode in modes:
        q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n, s=s)
        dev = get_mode_deviation(deviations, mode.l, mode.m, mode.n)

        omega_r_new, omega_i_new = deform_qnm_physical_parameters(
            q["omega_r_si"],
            q["omega_i_si"],
            dev,
        )

        amp_new = mode.amplitude * (1.0 + dev.d_amp_frac)
        phase_new = mode.phase + dev.d_phase

        h += component_strain_from_freqs(
            amplitude=amp_new,
            phase=phase_new,
            omega_r_si=omega_r_new,
            omega_i_si=omega_i_new,
            t=t,
            t0=t0,
        )

    return RingdownSignal(t=t, h_complex=h)

# ============================================================
# Block 2
#
# What this block does:
# 1. Defines a simple white-noise injection function
# 2. Defines a toy colored PSD
# 3. Defines time-domain overlap tools
# 4. Defines frequency-domain PSD-weighted inner products
# 5. Defines a matched-filter-like SNR
# ============================================================

# ------------------------------------------------------------
# White noise injection
# For now we inject only into h_plus.
# ------------------------------------------------------------
def add_white_noise(signal: RingdownSignal, sigma: float, seed: int | None = None) -> RingdownSignal:
    rng = np.random.default_rng(seed)
    noisy_plus = signal.h_plus + rng.normal(0.0, sigma, size=len(signal.t))
    noisy_complex = noisy_plus.astype(np.complex128)
    return RingdownSignal(t=signal.t.copy(), h_complex=noisy_complex)


# ------------------------------------------------------------
# Simple colored PSD
# This is a pedagogical PSD, not an official detector PSD.
# ------------------------------------------------------------
def analytic_psd(freqs: np.ndarray, f0: float = 250.0, floor: float = 1e-46) -> np.ndarray:
    f = np.maximum(freqs, 1.0)
    low = (f0 / f) ** 4
    mid = 1.0
    high = (f / (2.5 * f0)) ** 2
    return floor * (low + mid + high)


# ------------------------------------------------------------
# Time-domain overlap tools
# ------------------------------------------------------------
def waveform_inner_product(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    return float(np.sum(h1 * h2) * dt)

def waveform_norm(h: np.ndarray, dt: float) -> float:
    return np.sqrt(max(waveform_inner_product(h, h, dt), 0.0))

def waveform_overlap(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    n1 = waveform_norm(h1, dt)
    n2 = waveform_norm(h2, dt)
    if n1 == 0.0 or n2 == 0.0:
        return 0.0
    return waveform_inner_product(h1, h2, dt) / (n1 * n2)

def waveform_mismatch(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    return 1.0 - waveform_overlap(h1, h2, dt)


# ------------------------------------------------------------
# Frequency-domain PSD-weighted tools
# Convention:
#   <h1|h2> = 4 Re ∑ h1(f) h2*(f) / S_n(f) df
# ------------------------------------------------------------
def fd_inner(h1: np.ndarray, h2: np.ndarray, dt: float, psd: np.ndarray) -> float:
    h1f = np.fft.rfft(h1)
    h2f = np.fft.rfft(h2)
    freqs = np.fft.rfftfreq(len(h1), dt)

    if len(psd) != len(freqs):
        raise ValueError("psd must have the same length as the rfft frequency grid")

    df = freqs[1] - freqs[0]
    return float(4.0 * np.real(np.sum(h1f * np.conjugate(h2f) / psd) * df))

def fd_norm(h: np.ndarray, dt: float, psd: np.ndarray) -> float:
    return np.sqrt(max(fd_inner(h, h, dt, psd), 0.0))

def fd_overlap(h1: np.ndarray, h2: np.ndarray, dt: float, psd: np.ndarray) -> float:
    n1 = fd_norm(h1, dt, psd)
    n2 = fd_norm(h2, dt, psd)
    if n1 == 0.0 or n2 == 0.0:
        return 0.0
    return fd_inner(h1, h2, dt, psd) / (n1 * n2)

def fd_mismatch(h1: np.ndarray, h2: np.ndarray, dt: float, psd: np.ndarray) -> float:
    return 1.0 - fd_overlap(h1, h2, dt, psd)

def matched_filter_snr(template: np.ndarray, data: np.ndarray, dt: float, psd: np.ndarray) -> float:
    ntemp = fd_norm(template, dt, psd)
    if ntemp == 0.0:
        return 0.0
    return fd_inner(data, template, dt, psd) / ntemp

# ============================================================
# Block 3
#
# What this block does:
# 1. Builds fiducial GR and MG signals
# 2. Injects one of them into white noise
# 3. Builds a toy PSD
# 4. Computes time-domain and PSD-weighted overlaps
# 5. Computes matched-filter-like SNRs
# 6. Makes diagnostic plots
# ============================================================

# ------------------------------------------------------------
# Fiducial remnant and mode content
# ------------------------------------------------------------
remnant = KerrRemnant(mass_msun=60.0, chi=0.7)

modes = [
    RingdownMode(l=2, m=2, n=0, amplitude=1.00, phase=0.00),
    RingdownMode(l=2, m=2, n=1, amplitude=0.45, phase=0.30),
    RingdownMode(l=3, m=3, n=0, amplitude=0.25, phase=-0.50),
]

duration = 0.05
sample_rate = 8192
t0 = 0.002

# ------------------------------------------------------------
# GR baseline signal
# ------------------------------------------------------------
signal_gr = generate_gr_ringdown(
    remnant=remnant,
    modes=modes,
    duration=duration,
    sample_rate=sample_rate,
    t0=t0,
)

# ------------------------------------------------------------
# One example mode-dependent modified-gravity signal
# ------------------------------------------------------------
deviations = {
    (2, 2, 0): ModeDeviation(d_omega_frac=0.02,  d_tau_frac=0.10, d_amp_frac=0.00, d_phase=0.00),
    (2, 2, 1): ModeDeviation(d_omega_frac=-0.01, d_tau_frac=0.15, d_amp_frac=0.05, d_phase=0.10),
    (3, 3, 0): ModeDeviation(d_omega_frac=0.03,  d_tau_frac=-0.05, d_amp_frac=-0.10, d_phase=-0.08),
}

signal_mg = generate_modified_ringdown(
    remnant=remnant,
    modes=modes,
    deviations=deviations,
    duration=duration,
    sample_rate=sample_rate,
    t0=t0,
)

# ------------------------------------------------------------
# Inject a signal into white noise
# You can choose to inject either the GR or MG signal here.
# ------------------------------------------------------------
inject_mg = False

signal_injected = signal_mg if inject_mg else signal_gr
data = add_white_noise(signal_injected, sigma=0.03, seed=1)

# ------------------------------------------------------------
# Build the PSD on the frequency grid
# ------------------------------------------------------------
dt = data.t[1] - data.t[0]
freqs = np.fft.rfftfreq(len(data.t), dt)
psd = analytic_psd(freqs)

# ------------------------------------------------------------
# Time-domain diagnostics
# ------------------------------------------------------------
td_overlap_gr_mg = waveform_overlap(signal_gr.h_plus, signal_mg.h_plus, dt)
td_mismatch_gr_mg = waveform_mismatch(signal_gr.h_plus, signal_mg.h_plus, dt)

print("Time-domain waveform comparison")
print(f"Overlap(GR, MG)   = {td_overlap_gr_mg:.6f}")
print(f"Mismatch(GR, MG)  = {td_mismatch_gr_mg:.6f}")

# ------------------------------------------------------------
# Frequency-domain PSD-weighted diagnostics
# ------------------------------------------------------------
fd_overlap_gr_mg = fd_overlap(signal_gr.h_plus, signal_mg.h_plus, dt, psd)
fd_mismatch_gr_mg = fd_mismatch(signal_gr.h_plus, signal_mg.h_plus, dt, psd)

print("\nPSD-weighted waveform comparison")
print(f"FD Overlap(GR, MG)   = {fd_overlap_gr_mg:.6f}")
print(f"FD Mismatch(GR, MG)  = {fd_mismatch_gr_mg:.6f}")

# ------------------------------------------------------------
# Matched-filter-like SNRs against the noisy data
# ------------------------------------------------------------
snr_gr = matched_filter_snr(signal_gr.h_plus, data.h_plus, dt, psd)
snr_mg = matched_filter_snr(signal_mg.h_plus, data.h_plus, dt, psd)

print("\nMatched-filter-like SNRs on noisy data")
print(f"SNR using GR template = {snr_gr:.6e}")
print(f"SNR using MG template = {snr_mg:.6e}")

# ------------------------------------------------------------
# Plot data, GR template, MG template
# ------------------------------------------------------------
plt.figure()
plt.plot(data.t, data.h_plus, label="Noisy data", alpha=0.8)
plt.plot(signal_gr.t, signal_gr.h_plus, label="GR template")
plt.plot(signal_mg.t, signal_mg.h_plus, label="MG template")
plt.xlabel("Time [s]")
plt.ylabel(r"$h_+$ [arb.]")
plt.title("Injection into noise: data and competing templates")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Plot residuals with respect to each template
# ------------------------------------------------------------
residual_gr = data.h_plus - signal_gr.h_plus
residual_mg = data.h_plus - signal_mg.h_plus

plt.figure()
plt.plot(data.t, residual_gr, label="Residual using GR template")
plt.plot(data.t, residual_mg, label="Residual using MG template")
plt.xlabel("Time [s]")
plt.ylabel("Residual [arb.]")
plt.title("Template residuals")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Plot amplitude spectra
# ------------------------------------------------------------
data_fft = np.abs(np.fft.rfft(data.h_plus))
gr_fft = np.abs(np.fft.rfft(signal_gr.h_plus))
mg_fft = np.abs(np.fft.rfft(signal_mg.h_plus))

plt.figure()
plt.plot(freqs, data_fft, label="Noisy data")
plt.plot(freqs, gr_fft, label="GR template")
plt.plot(freqs, mg_fft, label="MG template")
plt.xlim(0, 1500)
plt.xlabel("Frequency [Hz]")
plt.ylabel("FFT amplitude [arb.]")
plt.title("Frequency-domain comparison")
plt.legend()
plt.show()

# ============================================================
# Block 4
#
# What this block does:
# 1. Scans a deformation-strength parameter lambda
# 2. Measures mismatch and SNR preference as lambda varies
# 3. Makes a 2D map over dominant-mode frequency and damping shifts
# 4. Stores arrays for later notebooks
# ============================================================

# ------------------------------------------------------------
# A helper that scales a reference deformation dictionary by lambda
# ------------------------------------------------------------
def scale_deviations(
    base_deviations: dict[tuple[int, int, int], ModeDeviation],
    lam: float,
) -> dict[tuple[int, int, int], ModeDeviation]:
    scaled = {}
    for key, dev in base_deviations.items():
        scaled[key] = ModeDeviation(
            d_omega_frac=lam * dev.d_omega_frac,
            d_tau_frac=lam * dev.d_tau_frac,
            d_amp_frac=lam * dev.d_amp_frac,
            d_phase=lam * dev.d_phase,
        )
    return scaled


# ------------------------------------------------------------
# 1D scan in total deformation strength
# lambda = 0 is GR
# lambda = 1 is the full example MG model
# ------------------------------------------------------------
lambda_vals = np.linspace(0.0, 1.5, 61)

td_mismatches = []
fd_mismatches = []
snr_with_gr = []
snr_with_mg = []

for lam in lambda_vals:
    trial_devs = scale_deviations(deviations, lam)

    signal_trial = generate_modified_ringdown(
        remnant=remnant,
        modes=modes,
        deviations=trial_devs,
        duration=duration,
        sample_rate=sample_rate,
        t0=t0,
    )

    td_mismatches.append(
        waveform_mismatch(signal_gr.h_plus, signal_trial.h_plus, dt)
    )

    fd_mismatches.append(
        fd_mismatch(signal_gr.h_plus, signal_trial.h_plus, dt, psd)
    )

    # Compare how well GR and the trial template match the same noisy data
    snr_with_gr.append(
        matched_filter_snr(signal_gr.h_plus, data.h_plus, dt, psd)
    )

    snr_with_mg.append(
        matched_filter_snr(signal_trial.h_plus, data.h_plus, dt, psd)
    )

td_mismatches = np.array(td_mismatches)
fd_mismatches = np.array(fd_mismatches)
snr_with_gr = np.array(snr_with_gr)
snr_with_mg = np.array(snr_with_mg)

# ------------------------------------------------------------
# Plot mismatch vs deformation strength
# ------------------------------------------------------------
plt.figure()
plt.plot(lambda_vals, td_mismatches, label="Time-domain mismatch")
plt.plot(lambda_vals, fd_mismatches, label="PSD-weighted mismatch")
plt.xlabel(r"Deformation strength $\lambda$")
plt.ylabel("Mismatch")
plt.title("Template mismatch vs deformation strength")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Plot template preference in terms of matched-filter SNR
# ------------------------------------------------------------
plt.figure()
plt.plot(lambda_vals, snr_with_gr, label="SNR with GR template")
plt.plot(lambda_vals, snr_with_mg, label="SNR with trial MG template")
plt.xlabel(r"Deformation strength $\lambda$")
plt.ylabel("Matched-filter-like SNR")
plt.title("Template preference vs deformation strength")
plt.legend()
plt.show()

# ------------------------------------------------------------
# Plot the difference in template score
# Positive values mean the trial MG template scores better.
# ------------------------------------------------------------
delta_snr = snr_with_mg - snr_with_gr

plt.figure()
plt.plot(lambda_vals, delta_snr)
plt.xlabel(r"Deformation strength $\lambda$")
plt.ylabel(r"$\mathrm{SNR}_{MG} - \mathrm{SNR}_{GR}$")
plt.title("Relative template preference")
plt.show()

# ------------------------------------------------------------
# 2D scan for the dominant mode only:
# vary d_omega_220 and d_tau_220 and see which template fits data better
# ------------------------------------------------------------
domega_vals = np.linspace(-0.04, 0.04, 41)
dtau_vals = np.linspace(-0.20, 0.20, 41)

fd_mismatch_grid = np.zeros((len(dtau_vals), len(domega_vals)))
delta_snr_grid = np.zeros((len(dtau_vals), len(domega_vals)))

for i, dtau220 in enumerate(dtau_vals):
    for j, domega220 in enumerate(domega_vals):
        trial_devs = {
            (2, 2, 0): ModeDeviation(
                d_omega_frac=float(domega220),
                d_tau_frac=float(dtau220),
                d_amp_frac=0.0,
                d_phase=0.0,
            )
        }

        signal_trial = generate_modified_ringdown(
            remnant=remnant,
            modes=modes,
            deviations=trial_devs,
            duration=duration,
            sample_rate=sample_rate,
            t0=t0,
        )

        fd_mismatch_grid[i, j] = fd_mismatch(signal_gr.h_plus, signal_trial.h_plus, dt, psd)

        snr_trial = matched_filter_snr(signal_trial.h_plus, data.h_plus, dt, psd)
        delta_snr_grid[i, j] = snr_trial - snr_gr

# ------------------------------------------------------------
# Plot PSD-weighted mismatch map
# ------------------------------------------------------------
plt.figure()
plt.imshow(
    fd_mismatch_grid,
    origin="lower",
    aspect="auto",
    extent=[domega_vals[0], domega_vals[-1], dtau_vals[0], dtau_vals[-1]],
)
plt.colorbar(label="PSD-weighted mismatch")
plt.xlabel(r"$\delta\omega_{220}$")
plt.ylabel(r"$\delta\tau_{220}$")
plt.title("PSD-weighted mismatch map for dominant-mode deformation")
plt.show()

# ------------------------------------------------------------
# Plot data-driven preference map
# Positive values mean that deformation matches the noisy data better than GR.
# ------------------------------------------------------------
plt.figure()
plt.imshow(
    delta_snr_grid,
    origin="lower",
    aspect="auto",
    extent=[domega_vals[0], domega_vals[-1], dtau_vals[0], dtau_vals[-1]],
)
plt.colorbar(label=r"$\mathrm{SNR}_{trial} - \mathrm{SNR}_{GR}$")
plt.xlabel(r"$\delta\omega_{220}$")
plt.ylabel(r"$\delta\tau_{220}$")
plt.title("Template preference map from noisy injection")
plt.show()

# ------------------------------------------------------------
# Save products for later notebooks
# ------------------------------------------------------------
np.savez(
    "template_validation_and_injections.npz",
    t=data.t,
    data_h_plus=data.h_plus,
    gr_h_plus=signal_gr.h_plus,
    mg_h_plus=signal_mg.h_plus,
    freqs=freqs,
    psd=psd,
    lambda_vals=lambda_vals,
    td_mismatches=td_mismatches,
    fd_mismatches=fd_mismatches,
    snr_with_gr=snr_with_gr,
    snr_with_mg=snr_with_mg,
    domega_vals=domega_vals,
    dtau_vals=dtau_vals,
    fd_mismatch_grid=fd_mismatch_grid,
    delta_snr_grid=delta_snr_grid,
)

results_bundle = {
    "remnant": asdict(remnant),
    "modes": [asdict(m) for m in modes],
    "reference_deviations": {str(k): asdict(v) for k, v in deviations.items()},
    "inject_mg": inject_mg,
    "snr_gr": float(snr_gr),
    "snr_mg": float(snr_mg),
}

print("Saved arrays to template_validation_and_injections.npz")
print("Results bundle:")
print(results_bundle)